In [1]:
!pip install requests beautifulsoup4 newspaper3k tldextract python-Levenshtein

  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached charset_normalizer-3.4.4-cp313-cp313-win_amd64.whl.metadata (38 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached certifi-2026.1.4-py3-none-any.whl.metadata (2.5 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-win_amd64.whl.metadata (2.4 kB)
  Using cached lxml-6.0.2-cp313-cp313-win_amd64.whl.metadata (3.7 kB)
  Using cached nltk-3.9.2-py3-none-any.whl.metadata (3.2 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
     ---------------------------------------- 0.0/7.

In [23]:
!pip install cloudscraper fake-useragent

  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl (54 kB)

   ---------------------------------------- 0/4 [pyparsing]
   ---------------------------------------- 0/4 [pyparsing]
   ---------------------------------------- 0/4 [pyparsing]
   ---------------------------------------- 0/4 [pyparsing]
   -------------------- ------------------- 2/4 [requests-toolbelt]
   -------------------- ------------------- 2/4 [requests-toolbelt]
   -------------------- ------------------- 2/4 [requests-toolbelt]
   -------------------- ------------------- 2/4 [requests-toolbelt]
   -------------------- ------------------- 2/4 [requests-toolbelt]
   -------------------- ------------------- 2/4 [requests-toolbelt]
   -------------------- ------------------- 2/4 [requests-toolbelt]
   -------------------- ------------------- 2/4 [requests-toolbelt]
   ------------------------------ --------- 3/4 [cloudscraper]
   ------

In [41]:
!pip install feedparser scikit-learn

  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   --- ------------------------------------ 0.8/8.0 MB 6.3 MB/s eta 0:00:02
   --------------- ------------------------ 3.1/8.0 MB 10.0 MB/s eta 0:00:01
   ---------------------------- ----------- 5.8/8.0 MB 10.9 MB/s eta 0:00:01
   ---------------------------------------  7.9/8.0 MB 10.9 MB/s eta 0:00:01
   ---------------------------------------  7.9/8.0 MB 10.9 MB/s eta 0:00:01
   ---------------------------------------- 8.0/8.0 MB 8.1 MB/s  0:00:01
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   --------- ------------------------------ 2.9/12.3 MB 14.4 MB/s eta 0:00:01
   ----------------- ---------------------- 5.5/12.3 MB 13.7 MB/s eta 0:00:01
   ---------------------- ----------------- 6.8/12.3 MB 13.6 MB/s eta 0:00:01
   ---------------------------------- ----- 10.5/12.3 MB 12.4 MB/s eta 0:00:01
   -------------------

In [24]:
import re
import tldextract
import Levenshtein
from bs4 import BeautifulSoup
from fake_useragent import UserAgent
import cloudscraper

In [42]:
import feedparser
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [49]:
VIETNAM_TRUSTED_DOMAINS = [
    "vnexpress.net",
    "tuoitre.vn",
    "thanhnien.vn",
    "vietnamnet.vn",
    "dantri.com.vn",
    "nhandan.vn",          # báo Đảng
    "chinhphu.vn",         # cổng thông tin Chính phủ
    "baochinhphu.vn"
]

INTERNATIONAL_TRUSTED_DOMAINS = [
    "bbc.com",
    "reuters.com",
    "apnews.com",
    "theguardian.com",
    "nytimes.com",
    "washingtonpost.com",
    "aljazeera.com"
]

TRUSTED_DOMAINS = VIETNAM_TRUSTED_DOMAINS + INTERNATIONAL_TRUSTED_DOMAINS

RULE_WEIGHTS = {
    "typosquatting": 0.25,   # mạo danh → cực kỳ nguy hiểm
    "domain": 0.20,          # uy tín nguồn
    "cross_source": 0.25,    # xác nhận đa nguồn
    "https": 0.08,
    "structure": 0.08,
    "content_trap": 0.08,
    "caps": 0.04,
    "length": 0.02
}

In [28]:
def fetch_html(url: str) -> str:
    ua = UserAgent()

    scraper = cloudscraper.create_scraper(
        browser={
            "browser": "chrome",
            "platform": "windows",
            "desktop": True
        }
    )

    headers = {
        "User-Agent": ua.random,
        "Accept-Language": "en-US,en;q=0.9,vi;q=0.8"
    }

    response = scraper.get(url, headers=headers, timeout=15)
    response.raise_for_status()
    return response.text

In [29]:
def extract_article(url: str):
    html = fetch_html(url)
    soup = BeautifulSoup(html, "html.parser")

    title = soup.title.text.strip() if soup.title else ""

    for tag in soup(["script", "style", "iframe", "noscript"]):
        tag.decompose()

    paragraphs = [
        p.get_text(" ", strip=True)
        for p in soup.find_all("p")
        if len(p.get_text(strip=True)) > 30
    ]

    text = "\n".join(paragraphs)

    return {
        "title": title,
        "text": text
    }

Rule Engine

In [30]:
def extract_domain(url: str) -> str:
    ext = tldextract.extract(url)
    return f"{ext.domain}.{ext.suffix}"

In [31]:
def rule_domain(url: str):
    domain = extract_domain(url)
    if domain in TRUSTED_DOMAINS:
        return 1.0, "Nguồn báo chính thống"
    return 0.6, "Nguồn không thuộc danh sách uy tín"

In [32]:
def rule_typosquatting(url: str, threshold=0.85):
    domain = extract_domain(url)
    for trusted in TRUSTED_DOMAINS:
        similarity = Levenshtein.ratio(domain, trusted)
        if similarity > threshold and domain != trusted:
            return 0.05, f"Domain mạo danh gần giống {trusted}"
    return 1.0, "Không phát hiện mạo danh domain"

In [33]:
def rule_https(url: str):
    if url.startswith("https://"):
        return 1.0, "Sử dụng HTTPS"
    return 0.4, "Không sử dụng HTTPS"

In [34]:
def rule_structure(article):
    if article["title"] and len(article["text"]) > 300:
        return 1.0, "Bài viết có cấu trúc rõ ràng"
    return 0.5, "Cấu trúc bài viết yếu"

In [35]:
def rule_length(article):
    word_count = len(article["text"].split())

    if word_count < 100:
        return 0.2, "Nội dung quá ngắn"
    if word_count < 300:
        return 0.5, "Nội dung thiếu ngữ cảnh"
    return 1.0, "Độ dài nội dung hợp lý"

In [36]:
def rule_caps(article):
    title = article["title"]
    if not title:
        return 0.6, "Không có tiêu đề"

    words = title.split()
    caps_ratio = sum(1 for w in words if w.isupper()) / max(len(words), 1)

    if caps_ratio > 0.3 or "!!!" in title or "???" in title:
        return 0.3, "Tiêu đề giật gân / cảm xúc mạnh"

    return 1.0, "Tiêu đề trung tính"

In [37]:
SHOCKING_WORDS = [
    "SỐC", "KHÔNG TIN NỔI", "RÙNG MÌNH",
    "KINH HOÀNG", "GÂY PHẪN NỘ"
]

AD_WORDS = [
    "mua ngay", "giá rẻ", "ưu đãi",
    "điều trị dứt điểm", "cam kết khỏi",
    "liên hệ ngay", "đặt hàng"
]

def rule_content_trap(article):
    text = (article["title"] + " " + article["text"]).upper()

    shock_hits = sum(1 for w in SHOCKING_WORDS if w in text)
    ad_hits = sum(1 for w in AD_WORDS if w.upper() in text)

    if shock_hits >= 2:
        return 0.3, "Nội dung giật gân cảm xúc"

    if ad_hits >= 2:
        return 0.2, "Dấu hiệu quảng cáo trá hình"

    return 1.0, "Không phát hiện bẫy nội dung"

AGGREGATOR – TÍNH ĐIỂM CUỐI

In [38]:
def aggregate_scores(rule_results: dict):
    weighted_sum = 0
    total_weight = 0

    for rule_name, (score, _) in rule_results.items():
        weight = RULE_WEIGHTS[rule_name]
        weighted_sum += score * weight
        total_weight += weight

    return round(weighted_sum / total_weight, 2)

CROSS SOURCE CHECK

In [43]:
def extract_claim(title: str) -> str:
    # Loại clickbait prefix
    title = re.sub(r"(SỐC|KHÔNG TIN NỔI|RÙNG MÌNH|KINH HOÀNG)\s*[:,!-]*", "", title, flags=re.IGNORECASE)
    return title.strip()

In [44]:
def google_news_search(query: str, limit=5):
    query = query.replace(" ", "+")
    url = f"https://news.google.com/rss/search?q={query}&hl=vi&gl=VN&ceid=VN:vi"

    feed = feedparser.parse(url)

    results = []
    for entry in feed.entries[:limit]:
        results.append({
            "title": entry.title,
            "link": entry.link
        })
    return results

In [45]:
def title_similarity(title, other_titles):
    texts = [title] + other_titles
    vectorizer = TfidfVectorizer().fit_transform(texts)
    vectors = vectorizer.toarray()

    similarities = cosine_similarity([vectors[0]], vectors[1:])[0]
    return similarities

In [47]:
def rule_cross_source(article, url):
    title = article["title"]
    claim = extract_claim(title)

    results = google_news_search(claim)

    if len(results) == 0:
        return 0.2, "Không tìm thấy nguồn độc lập xác nhận"

    other_titles = [r["title"] for r in results]
    similarities = title_similarity(title, other_titles)

    confirmed_sources = 0
    trusted_confirmations = 0
    original_domain = extract_domain(url)

    for sim, r in zip(similarities, results):
        if sim > 0.3:
            confirmed_sources += 1
            if extract_domain(r["link"]) in TRUSTED_DOMAINS:
                trusted_confirmations += 1

    if trusted_confirmations >= 1:
        return 1.0, "Có nguồn báo lớn xác nhận thông tin"

    if confirmed_sources >= 2:
        return 0.7, "Thông tin được nhiều nguồn độc lập đăng tải"

    return 0.4, "Chỉ có ít nguồn xác nhận"

In [48]:
def evaluate_url(url: str):
    article = extract_article(url)

    rule_results = {
        "typosquatting": rule_typosquatting(url),
        "domain": rule_domain(url),
        "https": rule_https(url),
        "structure": rule_structure(article),
        "length": rule_length(article),
        "caps": rule_caps(article),
        "content_trap": rule_content_trap(article),
        "cross_source": rule_cross_source(article, url)
    }

    final_score = aggregate_scores(rule_results)

    if final_score >= 0.8:
        label = "High trust"
    elif final_score >= 0.5:
        label = "Need verification"
    else:
        label = "High risk"

    return {
        "url": url,
        "score": final_score,
        "label": label,
        "rules": rule_results
    }

In [50]:
evaluate_url("https://blog-khoe-dep.blogspot.com/2013/10/lo-anh-mr-dam-nghich-chim-o-ho-boi.html")

{'url': 'https://blog-khoe-dep.blogspot.com/2013/10/lo-anh-mr-dam-nghich-chim-o-ho-boi.html',
 'score': 0.64,
 'label': 'Need verification',
 'rules': {'typosquatting': (1.0, 'Không phát hiện mạo danh domain'),
  'domain': (0.6, 'Nguồn không thuộc danh sách uy tín'),
  'https': (1.0, 'Sử dụng HTTPS'),
  'structure': (0.5, 'Cấu trúc bài viết yếu'),
  'length': (0.2, 'Nội dung quá ngắn'),
  'caps': (0.3, 'Tiêu đề giật gân / cảm xúc mạnh'),
  'content_trap': (1.0, 'Không phát hiện bẫy nội dung'),
  'cross_source': (0.2, 'Không tìm thấy nguồn độc lập xác nhận')}}

In [51]:
evaluate_url("https://vov.vn/the-gioi/nguoi-dan-cuba-lo-ngai-ve-khung-hoang-kinh-te-va-nang-luong-moi-post1259878.vov")

{'url': 'https://vov.vn/the-gioi/nguoi-dan-cuba-lo-ngai-ve-khung-hoang-kinh-te-va-nang-luong-moi-post1259878.vov',
 'score': 0.77,
 'label': 'Need verification',
 'rules': {'typosquatting': (1.0, 'Không phát hiện mạo danh domain'),
  'domain': (0.6, 'Nguồn không thuộc danh sách uy tín'),
  'https': (1.0, 'Sử dụng HTTPS'),
  'structure': (1.0, 'Bài viết có cấu trúc rõ ràng'),
  'length': (1.0, 'Độ dài nội dung hợp lý'),
  'caps': (1.0, 'Tiêu đề trung tính'),
  'content_trap': (1.0, 'Không phát hiện bẫy nội dung'),
  'cross_source': (0.4, 'Chỉ có ít nguồn xác nhận')}}